In [1]:


import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset as HFDataset

train_df = pd.read_csv("/kaggle/input/ererer/train.csv")  


label_encoder = LabelEncoder()
train_df["label"] = label_encoder.fit_transform(train_df["Subject"])
num_labels = len(label_encoder.classes_)


train_df_split, val_df_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=42
)


MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def preprocess_data(examples):
    return tokenizer(examples["Text"], truncation=True, padding="max_length", max_length=256)


train_dataset = HFDataset.from_pandas(train_df_split)
val_dataset = HFDataset.from_pandas(val_df_split)

train_dataset = train_dataset.map(preprocess_data, batched=True)
val_dataset = val_dataset.map(preprocess_data, batched=True)


train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    logging_steps=50 
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    f1 = f1_score(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)
    return {"f1": f1, "accuracy": acc}


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

eval_results = trainer.evaluate()
print(f"Validation Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"Validation F1 Score: {eval_results['eval_f1']:.4f}")


2025-08-09 15:54:14.919800: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754754854.941974     258 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754754854.948665     258 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_258/4044691539.py:81: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,1.356500,0.566722,0.875610,0.874000
2,0.561800,0.425400,0.896368,0.896000
3,0.401100,0.396542,0.908396,0.907500
4,0.341400,0.394023,0.906563,0.907000
5,0.315900,0.392734,0.908453,0.908000
6,0.279800,0.389908,0.913129,0.912500
7,0.275300,0.391136,0.912101,0.911500
8,0.263800,0.390780,0.911027,0.910500


Validation Accuracy: 0.9125
Validation F1 Score: 0.9131


In [3]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset as HFDataset

train_df = pd.read_csv("/kaggle/input/ererer/train.csv")
test_df = pd.read_csv("/kaggle/input/ererer/test.csv")

label_encoder = LabelEncoder()
train_df["label"] = label_encoder.fit_transform(train_df["Subject"])
num_labels = len(label_encoder.classes_)

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(examples):
    return tokenizer(examples["Text"], truncation=True, padding="max_length", max_length=256)

train_dataset = HFDataset.from_pandas(train_df[["Text", "label"]])
test_dataset = HFDataset.from_pandas(test_df)

train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels, ignore_mismatched_sizes=True)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    logging_steps=50 
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.tensor(logits).argmax(dim=1)
    f1 = f1_score(labels, preds, average="macro")
    return {"f1": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=train_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

predictions = trainer.predict(test_dataset)
pred_labels = predictions.predictions.argmax(axis=1)
pred_subjects = label_encoder.inverse_transform(pred_labels)

pd.DataFrame({"ID": test_df["ID"], "Subject": pred_subjects}).to_csv("submission.csv", index=False)


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4020 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1
1,0.511900,0.404376,0.910347
2,0.389700,0.309311,0.936796
3,0.341400,0.273717,0.946131
4,0.293900,0.245411,0.952678
5,0.246200,0.224949,0.956165
6,0.227400,0.205603,0.958484
7,0.208000,0.191451,0.961171
8,0.224300,0.177497,0.962494
9,0.189200,0.169720,0.963105
10,0.187400,0.165705,0.963701
